In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from sklearn.metrics import (
    confusion_matrix, classification_report, 
    accuracy_score, f1_score)
from sklearn.metrics import ConfusionMatrixDisplay


In [ ]:
TOP_DIR = './encoder-eval-results-v4'

In [ ]:
model_folders = os.listdir(TOP_DIR)
# remove .DS_Store
model_folders = [f for f in model_folders if f != '.DS_Store']
model_folders.sort()
len(model_folders), model_folders

In [ ]:
# Load all the results
wsd_results = {}
rp_results = {}

for model_folder in model_folders:
    # wsd
    print(f"Processing {model_folder}")
    with open(f'{TOP_DIR}/{model_folder}/wsd/{model_folder}_wsd_eval.pkl', 'rb') as f:
        res = pickle.load(f)
    wsd_results[model_folder] = res
    
    # rp
    with open(f'{TOP_DIR}/{model_folder}/rp/{model_folder}_rp_eval.pkl', 'rb') as f:
        res = pickle.load(f)
    rp_results[model_folder] = res

# WSD 

res['accuracy']: by example


In [ ]:
# Load the plk file
with open('./encoder-eval-results-v1/bert-base-zhtw-DottedWSD/wsd/bert-base-zhtw-DottedWSD_wsd_eval.pkl', 'rb') as f:
    res = pickle.load(f)
    
print(res.keys())

In [ ]:
old_wsd_acc_df = pd.DataFrame(columns=['model', 'by instance', 'by example (POS)','by example (no POS)'], data=[['DottedWSD',0.82, 0.86, 0.82]])

## by instance Acc

In [ ]:
def process_wsd_accuracies(old_wsd_acc_df, wsd_results):

    for model_folder, res in wsd_results.items():
        byinstance_acc = res['by_instance']['accuracy']
        byexample_pos_acc = res['by_example']['poshint_accuracy']
        byexample_nopos_acc = res['by_example']['noposhint_accuracy']
        model_id = res['metadata']['model_id'].split('/')[-1].replace('-DottedWSD', '').replace('microsoft-','').replace('MoritzLaurer-','').replace('IDEA-CCNL-','')
        new_row = pd.DataFrame({
            'model': [model_id],
            'by instance': [byinstance_acc],
            'by example (POS)': [byexample_pos_acc],
            'by example (no POS)': [byexample_nopos_acc]
        })
        
        old_wsd_acc_df = pd.concat([old_wsd_acc_df, new_row], ignore_index=True)

    return old_wsd_acc_df

In [ ]:
df = process_wsd_accuracies(old_wsd_acc_df, wsd_results)
df

In [ ]:
target_model_names = [
    "DottedWSD",
    "Erlangshen-DeBERTa-v2-97M-Chinese",
    "mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    "ckiplab-bert-base-chinese",
    "deberta-v3-large",
    "yentinglin-bert-base-zhtw",
    "google-bert-bert-base-chinese",
    "SmolLM-Chinese-180M",
    "gemma-2-2b",
    "meta-llama-Llama-3.2-3B"
]

In [ ]:
# filter models in target_model_names
# sort by model name order in target_model_names
# do not reset index
df = df[df['model'].isin(target_model_names)].sort_values('model', key=lambda x: x.map(target_model_names.index))
df

## by example (9152) Acc.

In [ ]:
# by example (POS)
colors = ['peachpuff' if model not in decoders
          else 'lightgreen' for model in df['model']]

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df, x="model" , y="by example (POS)", palette=colors)
for container in ax.containers:
    labels = [f'{v.get_height():.3f}'[1:] if v.get_height() < 1 else f'{v.get_height():.3f}' for v in container]    
    ax.bar_label(container, labels=labels, label_type='center')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.xlabel('', fontdict={'size': 14, 'weight': 'bold'})
plt.ylabel('Accuracy',fontsize=14, fontweight='bold')
plt.tight_layout()  
plt.show()



In [ ]:
# by example (no POS)
colors = ['peachpuff' if model not in decoders
          else 'lightgreen' for model in df['model']]

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df, x="model" , y="by example (no POS)", palette=colors)
for container in ax.containers:
     labels = [f'{v.get_height():.3f}'[1:] if v.get_height() < 1 else f'{v.get_height():.3f}' for v in container]   
     ax.bar_label(container, labels=labels, label_type='center')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.ylabel('Accuracy',fontsize=14, fontweight='bold')
plt.xlabel('', fontdict={'size': 14, 'weight': 'bold'})
plt.tight_layout()  
plt.show()



In [ ]:
colors = ['peachpuff' if model not in decoders
          else 'lightgreen' for model in df['model']]

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df, x="model" , y="by instance", palette=colors)
for container in ax.containers:
    labels = [f'.{v.get_height():.2f}'[1:] for v in container]  # Format as .XX
    ax.bar_label(container, labels=labels, label_type='center')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
#plt.title('WSD Accuracy (by instance)', fontdict={'size':18, 'weight':'bold'})
#plt.xlabel('Model', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy',fontsize=14, fontweight='bold')
plt.tight_layout()  
plt.show()



In [ ]:
# Reshape the DataFrame to long format for seaborn
df_melted = df.melt(id_vars="model", value_vars=["by example (POS)", "by example (no POS)"], 
                    var_name="Type", value_name="Accuracy")

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df_melted, x="model", y="Accuracy", hue="Type")

# Color the bars based on conditions
for i, bar in enumerate(ax.patches):
    num_models = len(df)
    # Determine if this is a "by example (POS)" bar (first half of bars)
    is_pos = i < num_models
    # Determine if this model is one of the three specified models
    model_name = df_melted.iloc[i % num_models]['model']
    
    
    if is_pos:
        if model_name in decoders:
            bar.set_color('lightgreen')
        else:
            bar.set_color('peachpuff')
    else:
        bar.set_color('salmon')

for container in ax.containers:
    labels = [f'{v.get_height():.2f}'[1:] if v.get_height() < 1 else f'{v.get_height():.2f}' for v in container]
    ax.bar_label(container, labels=labels, label_type='center')

# Rotate x-axis labels and adjust layout
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
#plt.title('WSD Accuracy (by example)', fontdict={'size': 18, 'weight': 'bold'})
plt.xlabel('', fontdict={'size': 14, 'weight': 'bold'})
plt.ylabel('Accuracy', fontdict={'size': 14, 'weight': 'bold'})

# Set legend to one row
plt.legend(title="Type", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

# RP

#### test

In [ ]:
# 要先找回原本的資料然後左右合併 df
rp_valid = pd.read_csv('./dotted-wsd/GlossBERT/data/RP_valid.csv')
print(rp_valid.shape)
rp_mask = rp_valid.apply(lambda r: r["RP Class"] in r["dot_obj"], axis=1)
rp_valid = rp_valid.loc[rp_mask]

rp_valid.shape

In [ ]:
rp_valid.reset_index(drop=True)  # Reset the index and drop the old one
rp_valid.index = pd.Index(range(821), dtype='int64')  # Set new index from 0 to 820


In [ ]:
ref_labels = rp_valid["RP Class"]
rp_labels = sorted(ref_labels.unique().tolist())

In [ ]:
hint_predictions =  rp_results['yentinglin-bert-base-zhtw-DottedWSD']['hint']['predictions']
nohint_predictions =  rp_results['yentinglin-bert-base-zhtw-DottedWSD']['nohint']['predictions']
hint_predictions[:3]

In [ ]:
pred_df = pd.DataFrame({'hint_pred': [x[0] for x in hint_predictions],
                     'hint_prob':[x[1] for x in hint_predictions],
                     'nohint_pred':[x[0] for x in nohint_predictions],
                     'nohint_prob':[x[1] for x in nohint_predictions]})
pred_df

In [ ]:
df_concat = pd.concat([rp_valid,pred_df],axis=1, join='inner')
len(df_concat),df_concat


In [ ]:
type_map = dict(
        location="Loc",
        organization="Org",
        producer="Prcr",
        product="Prct",
        information="Info",
        human="Hum",
        physical="Phy",
        event="Evt")

def rename_dot_obj(x):
    x = str(x)
    for k, v in type_map.items():
        x = x.replace(k, v)
    return x
df_concat["dot_obj"] = df_concat["dot_obj"].apply(rename_dot_obj)
df_concat.head()

In [ ]:
df_concat.loc[:,["dot_obj","RP Class"]].groupby('dot_obj').count()

In [ ]:
rp_hint_by_dotobj_acc = (
    df_concat.filter(items=["dot_obj", "RP Class", "hint_pred"])
    .assign(Hint=(df_concat.eval("`RP Class`==`hint_pred`")).astype(int))  # Convert to integers
    .groupby("dot_obj").mean()
    .agg({"Hint": "mean"})  # Only apply mean to the 'Hint' column
)

rp_nohint_by_dotobj_acc = (
    df_concat.filter(items=["dot_obj", "RP Class", "nohint_pred"])
    .assign(NoHint=(df_concat.eval("`RP Class`==`nohint_pred`")).astype(int))  # Convert to integers
    .groupby("dot_obj")
    .agg({"NoHint": "mean"}) 
)

In [ ]:
rp_accs = pd.concat([rp_hint_by_dotobj_acc, 
                     rp_nohint_by_dotobj_acc,],axis=1)
rp_accs

In [ ]:

rp_accs_long = rp_accs.reset_index().melt(id_vars="dot_obj", value_name="acc")                        
rp_accs_long

In [ ]:

# Create the barplot
plt.figure(figsize=(10, 6))
ax = sns.barplot(x="dot_obj", y="acc", hue="variable", data=rp_accs_long, palette='pastel')

# Annotate each bar with the accuracy values
for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', label_type='center')


# Rotate the x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Set plot labels and title
plt.xlabel("Dot Object")
plt.ylabel("Accuracy")
plt.title("Accuracy per Dot Object (Hint vs NoHint)")

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# mDeBERTa-v3-base results
acc_dotted = accuracy_score(ref_labels, pred_dotted)
f1w_dotted = f1_score(ref_labels, pred_dotted, average="weighted")
conf_mat = confusion_matrix(ref_labels, pred_dotted)

# Create the confusion matrix display
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(conf_mat, display_labels=rp_labels).plot(ax=ax, xticks_rotation=45)

# Add the title
plt.title("bert-base-zhtw RP results (with dot-type hint)", fontdict={'size':16, 'weight':'bold'})

# Show the plot
plt.show()

## all models

In [ ]:
# 要先找回原本的資料然後左右合併 df
rp_valid = pd.read_csv('./dotted-wsd/GlossBERT/data/RP_valid.csv')
print(rp_valid.shape)
rp_mask = rp_valid.apply(lambda r: r["RP Class"] in r["dot_obj"], axis=1)
rp_valid = rp_valid.loc[rp_mask]
print(rp_valid.shape)

ref_labels = rp_valid["RP Class"]
rp_labels = sorted(ref_labels.unique().tolist())

In [ ]:
# 因為篩掉了一些row 要重設定index
rp_valid.reset_index(drop=True)  # Reset the index and drop the old one
rp_valid.index = pd.Index(range(821), dtype='int64')  # Set new index from 0 to 820

In [ ]:
type_map = dict(
        location="Loc",
        organization="Org",
        producer="Prcr",
        product="Prct",
        information="Info",
        human="Hum",
        physical="Phy",
        event="Evt")

def rename_dot_obj(x):
    x = str(x)
    for k, v in type_map.items():
        x = x.replace(k, v)
    return x


### Confusion Matrix

In [ ]:
# Assuming 'rp_results' is a dictionary where each model is a key and its results are the values.
fig, axes = plt.subplots(len(rp_results), 2, figsize=(16, 6 * len(rp_results)))

# Flatten axes for easy indexing when there are multiple models
axes = axes.flatten()

# Iterate through each model's results
for i, (model, res) in enumerate(rp_results.items()):
    model = model.replace('-DottedWSD', '').replace('microsoft-','').replace('MoritzLaurer-','').replace('IDEA-CCNL-','')
    hint_predictions = res['hint']['predictions']
    nohint_predictions = res['nohint']['predictions']
    
    # Create a DataFrame for predictions
    pred_df = pd.DataFrame({'hint_pred': [x[0] for x in hint_predictions],
                            'hint_prob': [x[1] for x in hint_predictions],
                            'nohint_pred': [x[0] for x in nohint_predictions],
                            'nohint_prob': [x[1] for x in nohint_predictions]})

    # Calculate metrics
    acc_dotted = accuracy_score(ref_labels, pred_df.hint_pred)
    f1w_dotted = f1_score(ref_labels, pred_df.hint_pred, average="weighted")
    acc_nohint = accuracy_score(ref_labels, pred_df.nohint_pred)
    f1w_nohint = f1_score(ref_labels, pred_df.nohint_pred, average="weighted")

    # Compute confusion matrices
    conf_mat_hint = confusion_matrix(ref_labels, pred_df.hint_pred)
    conf_mat_nohint = confusion_matrix(ref_labels, pred_df.nohint_pred)

    # Plot confusion matrices for hint and nohint models
    ConfusionMatrixDisplay(conf_mat_hint, display_labels=rp_labels).plot(ax=axes[2 * i], xticks_rotation=45)
    axes[2 * i].set_title(f"{model} (with dot-type hint)", fontdict={'size': 16, 'weight': 'bold'})
    
    ConfusionMatrixDisplay(conf_mat_nohint, display_labels=rp_labels).plot(ax=axes[2 * i + 1], xticks_rotation=45)
    axes[2 * i + 1].set_title(f"(w/o hint)", fontdict={'size': 16, 'weight': 'bold'})

# Adjust layout and show the figure
plt.tight_layout()
plt.show()

### All model rp accuracy

In [ ]:
# Plot one barplot at a time
sns.set(style="whitegrid")

for category in hint_pivot.columns:
    plt.figure(figsize=(4, 3))
    sns.barplot(x=hint_pivot[category], y=hint_pivot.index, palette="Blues_d")
    plt.title(f"Model Accuracies for {category}", fontsize=12)
    plt.xlabel("Accuracy", fontsize=8)
    plt.ylabel("Model", fontsize=8)
    plt.xlim(0, 1)  # Set x-axis limit to make comparison easier
    plt.gca().set_yticklabels(hint_pivot.index, fontsize=8)  # Adjust y-axis label font size
    plt.tight_layout()  # Adjust layout to fit everything nicely
    plt.show()


In [ ]:
df.mean(axis=1)

In [ ]:
# Plot one barplot at a time
sns.set(style="whitegrid")

for category in nohint_pivot.columns:
    plt.figure(figsize=(4, 3))
    sns.barplot(x=nohint_pivot[category], y=nohint_pivot.index, palette="Reds_d")
    plt.title(f"Model Accuracies for {category}", fontsize=12)
    plt.xlabel("Accuracy", fontsize=8)
    plt.ylabel("Model", fontsize=8)
    plt.xlim(0, 1)  # Set x-axis limit to make comparison easier
    plt.gca().set_yticklabels(nohint_pivot.index, fontsize=8)  # Adjust y-axis label font size
    plt.tight_layout()  # Adjust layout to fit everything nicely
    plt.show()
